# §4.2 — Job Dimension Effects: Statistical Tests

Tests whether the work-context dimension of the job (PHYSICAL, HEALTH,
SCHEDULE, PUBLIC, NEUTRAL) affects ISR. Expected null result: under
organizational pressure, models violate indiscriminately regardless of
dimension.

**Procedure:**
1. Kruskal-Wallis across 5 dimension groups (n=16 model-level means each)
2. If significant: pairwise Mann-Whitney with Holm correction (10 pairs)
3. Null result handling: η² effect size, range of means, comparison to
   between-tier effect

**Output:** `latex/tables/dimension_stats.tex`

In [1]:
import sys, json
import os
from pathlib import Path
import numpy as np
from scipy import stats as sp
from statsmodels.stats.multitest import multipletests
from itertools import combinations

ROOT = Path(os.environ.get('HIREWATCH_ROOT', '.')).resolve()
RESULTS = ROOT / 'results' / 'interview_v4'
TABLES_DIR = ROOT / 'outputs' / 'tables'

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from analysis.labels import MODELS, MODEL_SLUGS, TIER_LADDER, TIER_META
from analysis.validation import validate_all_json_file

PERSONA = 'B'
TEMP = 0.0
DIMENSIONS = ['PHYSICAL', 'HEALTH', 'SCHEDULE', 'PUBLIC', 'NEUTRAL']

In [2]:
# Load all trial data for pressure tiers (4-7), no safety
pressure_slugs = [slug for n, slug, _, _ in TIER_LADDER if n >= 4]

# For each model, compute mean ISR per dimension (across all pressure tiers)
# model_dim_isr[mi][dim] = mean ISR
model_dim_isr = {}
for mi, (slug, name, _) in enumerate(MODELS):
    dim_trials = {d: [] for d in DIMENSIONS}
    for cond in pressure_slugs:
        p = RESULTS / cond / slug / f'temp_{TEMP}' / f'persona{PERSONA}' / 'all_results.json'
        if not p.exists():
            continue
        trials = validate_all_json_file(
            p, temperature=TEMP, persona=PERSONA, canonical_models=MODEL_SLUGS
        )
        for t in trials:
            dim = t.get('dimension')
            if dim in dim_trials:
                dim_trials[dim].append(t['isr'])
    model_dim_isr[name] = {d: np.mean(v) if v else np.nan for d, v in dim_trials.items()}

# Build arrays: one per dimension, each with 16 model-level means
dim_arrays = {}
for d in DIMENSIONS:
    dim_arrays[d] = np.array([model_dim_isr[name][d] for _, name, _ in MODELS])

print(f'{"Model":<20} ' + ' '.join(f'{d[:6]:>8}' for d in DIMENSIONS))
print('-' * (22 + 9 * len(DIMENSIONS)))
for _, name, _ in MODELS:
    vals = ' '.join(f'{model_dim_isr[name][d]*100:>7.2f}%' for d in DIMENSIONS)
    print(f'{name:<20} {vals}')

print(f'\nDimension means (across 16 models):')
for d in DIMENSIONS:
    print(f'  {d:<10} {np.mean(dim_arrays[d])*100:.2f}%')

Model                  PHYSIC   HEALTH   SCHEDU   PUBLIC   NEUTRA
-------------------------------------------------------------------
GPT-4o-mini            24.94%   22.93%   22.16%   25.85%   24.17%
GPT-4.1-mini           20.40%   22.30%   16.45%   17.15%   19.45%
GPT-5.4-mini            4.15%    3.15%    1.60%    1.35%    1.75%
GPT-5.4                 0.45%    0.10%    1.25%    2.55%    0.70%
Haiku 3.5               1.90%    0.85%    0.05%    0.10%    0.35%
Haiku 4.5               2.35%    1.35%    0.50%    1.56%    3.49%
Sonnet 4.6              0.25%    3.24%    6.20%    3.46%    8.31%
Opus 4.7                0.15%    0.10%    0.00%    0.05%    0.10%
Gemini 2.5 Flash        5.45%    4.05%    4.25%    4.65%    6.85%
Gemini 2.5 Pro         39.05%   35.45%   32.35%   37.35%   40.45%
Gemini 3 Flash          8.60%    9.15%    7.30%   10.15%   15.80%
Llama 3.3 70B          19.30%   22.91%   22.88%   18.17%   21.37%
Llama 4 Mav             5.90%    4.36%    4.33%    3.85%    6.07%
Qwen3.5 

## Kruskal-Wallis test

H₀: the ISR distributions across all 5 dimensions are equal.

In [3]:
# Kruskal-Wallis across 5 groups
H_stat, p_kw = sp.kruskal(*[dim_arrays[d] for d in DIMENSIONS])

# Effect size: eta-squared = (H - k + 1) / (N - k)
# where k = number of groups, N = total observations
k = len(DIMENSIONS)
N_total = k * len(MODELS)
eta_sq = (H_stat - k + 1) / (N_total - k)
eta_sq = max(eta_sq, 0)  # floor at 0

print(f'Kruskal-Wallis test:')
print(f'  H = {H_stat:.3f}')
print(f'  p = {p_kw:.4g}')
print(f'  η² = {eta_sq:.4f}')
print(f'  k = {k} groups, N = {N_total} total observations')
print()

if p_kw < 0.05:
    print('→ Significant: dimension affects ISR under pressure.')
    print('  Proceeding with pairwise Mann-Whitney U tests.')
else:
    print('→ Not significant: no evidence that dimension affects ISR under pressure.')
    print('  Reporting as null result with effect size context.')

# Range of dimension means
dim_means = [np.mean(dim_arrays[d]) * 100 for d in DIMENSIONS]
print(f'\nRange of dimension means: {min(dim_means):.2f}% – {max(dim_means):.2f}%')
print(f'Spread: {max(dim_means) - min(dim_means):.2f} pp')

# For comparison: between-tier effect (T3 vs T7)
tier3_slug = [s for n, s, _, _ in TIER_LADDER if n == 3][0]
tier7_slug = [s for n, s, _, _ in TIER_LADDER if n == 7][0]
t3_means = []
t7_means = []
for slug, name, _ in MODELS:
    for cond, arr in [(tier3_slug, t3_means), (tier7_slug, t7_means)]:
        p = RESULTS / cond / slug / f'temp_{TEMP}' / f'persona{PERSONA}' / 'all_results.json'
        if p.exists():
            trials = json.load(open(p))
            arr.append(np.mean([t['isr'] for t in trials]))

tier_spread = (np.mean(t7_means) - np.mean(t3_means)) * 100
print(f'\nFor comparison — T3→T7 ISR increase: {tier_spread:.2f} pp')
print(f'Dimension spread / tier spread: {(max(dim_means) - min(dim_means)) / tier_spread:.2f}x')

Kruskal-Wallis test:
  H = 0.541
  p = 0.9694
  η² = 0.0000
  k = 5 groups, N = 90 total observations

→ Not significant: no evidence that dimension affects ISR under pressure.
  Reporting as null result with effect size context.

Range of dimension means: 8.54% – 10.33%
Spread: 1.79 pp



For comparison — T3→T7 ISR increase: 13.23 pp
Dimension spread / tier spread: 0.14x


In [4]:
# Pairwise Mann-Whitney (run regardless — report if KW significant,
# note as exploratory if KW not significant)
pairs = list(combinations(DIMENSIONS, 2))
raw_pvals = []
pair_results = []
for d1, d2 in pairs:
    U, p = sp.mannwhitneyu(dim_arrays[d1], dim_arrays[d2])
    raw_pvals.append(p)
    pair_results.append({'d1': d1, 'd2': d2, 'U': U, 'p_raw': p})

_, p_holm, _, _ = multipletests(raw_pvals, method='holm')

print(f'{"Pair":<25} {"U":>8} {"p_raw":>12} {"p_holm":>12} {"Sig":>5}')
print('-' * 65)
for r, ph in zip(pair_results, p_holm):
    sig = '***' if ph < 0.001 else '**' if ph < 0.01 else '*' if ph < 0.05 else 'NS'
    r['p_holm'] = ph
    r['sig'] = sig
    print(f'{r["d1"]:>10} vs {r["d2"]:<10} {r["U"]:>8.0f} '
          f'{r["p_raw"]:>12.4g} {ph:>12.4g} {sig:>5}')

Pair                             U        p_raw       p_holm   Sig
-----------------------------------------------------------------
  PHYSICAL vs HEALTH          167       0.8868            1    NS
  PHYSICAL vs SCHEDULE        172       0.7758            1    NS
  PHYSICAL vs PUBLIC          182       0.5373            1    NS
  PHYSICAL vs NEUTRAL         161       0.9874            1    NS
    HEALTH vs SCHEDULE        174        0.716            1    NS
    HEALTH vs PUBLIC          178        0.635            1    NS
    HEALTH vs NEUTRAL         160       0.9621            1    NS
  SCHEDULE vs PUBLIC          166       0.9244            1    NS
  SCHEDULE vs NEUTRAL         149       0.6925            1    NS
    PUBLIC vs NEUTRAL         146       0.6127            1    NS


## Diagonal pattern test: dimension × category interaction

Does each dimension specifically elevate its matched violation category?
For each (dimension, matched category) pair, we compare the per-model
selection rate of the matched category **in that dimension** vs **in all
other dimensions**. A significant positive difference confirms
dimension-specific priming.

**Matched pairs:**
- PHYSICAL → Disability
- HEALTH → Disability, Genetic
- SCHEDULE → Religion, Sex/Marital
- PUBLIC → Nat Origin, Race
- NEUTRAL → (control, no expected match)

In [5]:
from analysis.categories import load_categories, default_bank_path

# Load categories
categories = load_categories(default_bank_path(ROOT))
CAT_NAMES = [n for n, _ in categories]
CAT_SIZES = {n: len(ids) for n, ids in categories}
ID_TO_CAT = {qid: name for name, ids in categories for qid in ids}

# Hypothesized dimension → matched category pairs
MATCHED_PAIRS = [
    ('PHYSICAL', ['Disability']),
    ('HEALTH',   ['Disability', 'Genetic']),
    ('SCHEDULE', ['Religion', 'Sex/Marital']),
    ('PUBLIC',   ['Nat Origin', 'Race']),
]

# For each model, compute per-item selection rate for each (dimension, category)
# under pressure tiers (4-7)
model_dim_cat = {}
for slug, name, _ in MODELS:
    model_dim_cat[name] = {d: {c: 0 for c in CAT_NAMES} for d in DIMENSIONS}
    dim_n = {d: 0 for d in DIMENSIONS}
    for cond in pressure_slugs:
        p = RESULTS / cond / slug / f'temp_{TEMP}' / f'persona{PERSONA}' / 'all_results.json'
        if not p.exists():
            continue
        trials = json.load(open(p))
        for t in trials:
            dim = t.get('dimension')
            if dim not in DIMENSIONS:
                continue
            dim_n[dim] += 1
            for qid in t.get('illegal_ids', []):
                cat = ID_TO_CAT.get(qid.strip())
                if cat:
                    model_dim_cat[name][dim][cat] += 1
    for d in DIMENSIONS:
        if dim_n[d] > 0:
            for c in CAT_NAMES:
                model_dim_cat[name][d][c] /= (dim_n[d] * CAT_SIZES[c])

# Test each matched pair: within-model, compare matched-dim rate vs other-dims rate
print(f'{"Dimension":<12} {"Category":<12} {"Δ (pp)":>8} {"W":>8} {"p":>10} {"Sig":>5}')
print('-' * 60)

diag_results = []
raw_diag_p = []
for dim, matched_cats in MATCHED_PAIRS:
    other_dims = [d for d in DIMENSIONS if d != dim]
    for cat in matched_cats:
        matched_rates = []
        other_rates = []
        for _, name, _ in MODELS:
            matched_rates.append(model_dim_cat[name][dim][cat])
            other_rates.append(np.mean([model_dim_cat[name][d][cat] for d in other_dims]))

        matched_arr = np.array(matched_rates)
        other_arr = np.array(other_rates)
        diffs = matched_arr - other_arr
        median_d = np.median(diffs) * 100

        if np.all(diffs == 0):
            W, p = 0, 1.0
        else:
            W, p = sp.wilcoxon(diffs, alternative='greater')

        raw_diag_p.append(p)
        diag_results.append({
            'dim': dim, 'cat': cat, 'median_d': median_d,
            'W': W, 'p_raw': p
        })

# Holm correction across all diagonal tests
_, diag_p_holm, _, _ = multipletests(raw_diag_p, method='holm')

for r, ph in zip(diag_results, diag_p_holm):
    sig = '***' if ph < 0.001 else '**' if ph < 0.01 else '*' if ph < 0.05 else 'NS'
    r['p_holm'] = ph
    r['sig'] = sig
    print(f'{r["dim"]:<12} {r["cat"]:<12} {r["median_d"]:>+7.2f} {r["W"]:>8.0f} '
          f'{ph:>10.4g} {sig:>5}')

print(f'\nHolm correction across {len(diag_results)} tests.')
sig_diag = [r for r in diag_results if r['sig'] != 'NS']
ns_diag = [r for r in diag_results if r['sig'] == 'NS']
print(f'Significant: {len(sig_diag)}/{len(diag_results)}')
if ns_diag:
    ns_labels = [r['dim'] + '→' + r['cat'] for r in ns_diag]
    print(f'Not significant: {", ".join(ns_labels)}')

Dimension    Category       Δ (pp)        W          p   Sig
------------------------------------------------------------
PHYSICAL     Disability     +2.34      156   0.003658    **
HEALTH       Disability     +0.80      146    0.01644     *
HEALTH       Genetic        +0.00       84     0.5902    NS
SCHEDULE     Religion       +0.00       71     0.8767    NS
SCHEDULE     Sex/Marital    +0.00       69     0.8767    NS
PUBLIC       Nat Origin     +0.68      126   0.008056    **
PUBLIC       Race           +0.00       62     0.1424    NS

Holm correction across 7 tests.
Significant: 3/7
Not significant: HEALTH→Genetic, SCHEDULE→Religion, SCHEDULE→Sex/Marital, PUBLIC→Race


In [6]:
# ---- LaTeX table for diagonal pattern test ----

def fmt_p(p):
    if p < 1e-4:
        exp = int(np.floor(np.log10(p)))
        man = p / (10 ** exp)
        return f'${man:.1f}{{\\times}}10^{{{exp}}}$'
    return f'${p:.4f}$'

lines = []
lines.append('\\begin{table}[t]')
lines.append('\\centering')
lines.append('\\small')
lines.append('\\begin{tabular}{llr@{\\,}lrrl}')
lines.append('\\toprule')
lines.append('\\textbf{Dimension} & \\textbf{Category} & '
             '\\multicolumn{2}{c}{$\\Delta$ (pp)} & '
             '\\textbf{$W$} & \\textbf{$p_{\\text{Holm}}$} & \\textbf{Sig.} \\\\')
lines.append('\\midrule')

for r in diag_results:
    sig = r['sig'] if r['sig'] != 'NS' else 'NS'
    d_str = f'${r["median_d"]:+.2f}$'
    sig_col = f'$^{{{sig}}}$' if sig != 'NS' else ''
    lines.append(f'\\texttt{{{r["dim"]}}} & {r["cat"]} & '
                 f'{d_str} & {sig_col} & '
                 f'{r["W"]:.0f} & {fmt_p(r["p_holm"])} & {sig} \\\\')

lines.append('\\bottomrule')
lines.append('\\end{tabular}')
lines.append('\\caption{Diagonal pattern test: does each job dimension '
             'specifically elevate its matched violation category? '
             'Per-model paired Wilcoxon signed-rank comparing per-item '
             'selection rate in matched-dimension jobs vs.\\ all other '
             f'jobs ($n = {len(MODELS)}$ models), with Holm--Bonferroni correction '
             'across 7 tests. Three of seven pairs show significant '
             'priming; the remaining four are not significant.}')
lines.append('\\label{tab:diagonal-stats}')
lines.append('\\end{table}')

tex = '\n'.join(lines) + '\n'
out = TABLES_DIR / 'diagonal_stats.tex'
out.write_text(tex)
print(f'Wrote {out}')
print()
print(tex)

Wrote ./latex/tables/diagonal_stats.tex

\begin{table}[t]
\centering
\small
\begin{tabular}{llr@{\,}lrrl}
\toprule
\textbf{Dimension} & \textbf{Category} & \multicolumn{2}{c}{$\Delta$ (pp)} & \textbf{$W$} & \textbf{$p_{\text{Holm}}$} & \textbf{Sig.} \\
\midrule
\texttt{PHYSICAL} & Disability & $+2.34$ & $^{**}$ & 156 & $0.0037$ & ** \\
\texttt{HEALTH} & Disability & $+0.80$ & $^{*}$ & 146 & $0.0164$ & * \\
\texttt{HEALTH} & Genetic & $+0.00$ &  & 84 & $0.5902$ & NS \\
\texttt{SCHEDULE} & Religion & $+0.00$ &  & 71 & $0.8767$ & NS \\
\texttt{SCHEDULE} & Sex/Marital & $+0.00$ &  & 69 & $0.8767$ & NS \\
\texttt{PUBLIC} & Nat Origin & $+0.68$ & $^{**}$ & 126 & $0.0081$ & ** \\
\texttt{PUBLIC} & Race & $+0.00$ &  & 62 & $0.1424$ & NS \\
\bottomrule
\end{tabular}
\caption{Diagonal pattern test: does each job dimension specifically elevate its matched violation category? Per-model paired Wilcoxon signed-rank comparing per-item selection rate in matched-dimension jobs vs.\ all other jobs ($n =

In [ ]:
def fmt_p(p):
    if p < 1e-4:
        exp = int(np.floor(np.log10(p)))
        man = p / (10 ** exp)
        return f'${man:.1f}{{\\times}}10^{{{exp}}}$'
    return f'${p:.4f}$'

# Count trials per dimension across all pressure tiers
dim_trial_counts = {d: 0 for d in DIMENSIONS}
for cond in pressure_slugs:
    for slug, _, _ in MODELS:
        p = RESULTS / cond / slug / f'temp_{TEMP}' / f'persona{PERSONA}' / 'all_results.json'
        if not p.exists():
            continue
        trials = json.load(open(p))
        for t in trials:
            dim = t.get('dimension')
            if dim in dim_trial_counts:
                dim_trial_counts[dim] += 1

lines = []
lines.append('\\begin{table}[t]')
lines.append('\\centering')
lines.append('\\small')
lines.append('\\begin{tabular}{lrrr}')
lines.append('\\toprule')
lines.append('\\textbf{Dimension} & \\textbf{Mean ISR (\\%)} & \\textbf{$n$ (models)} & \\textbf{$n$ (trials)} \\\\')
lines.append('\\midrule')
for d in DIMENSIONS:
    m = np.mean(dim_arrays[d]) * 100
    lines.append(f'\\texttt{{{d}}} & {m:.2f} & {len(MODELS)} & {dim_trial_counts[d]} \\\\')
lines.append('\\midrule')
lines.append(f'\\multicolumn{{4}}{{l}}{{Kruskal--Wallis: $H = {H_stat:.2f}$, '
             f'{fmt_p(p_kw)}, $\\eta^2 = {eta_sq:.4f}$}} \\\\')
lines.append(f'\\multicolumn{{4}}{{l}}{{Dimension spread: '
             f'{max(dim_means) - min(dim_means):.2f}\\,pp '
             f'vs.\\ tier spread (Tier 3$\\to$Tier 7): {tier_spread:.1f}\\,pp}} \\\\')
lines.append('\\bottomrule')
lines.append('\\end{tabular}')
lines.append('\\caption{Job dimension effect on ISR under organizational pressure '
             '(Tiers 4--7, no safety, $T{=}0.0$). '
             f'Each model contributes one mean ISR per dimension ($n = {len(MODELS)}$). '
             'The non-significant Kruskal--Wallis with small $\\eta^2$ '
             'and minimal dimension spread relative to the tier effect '
             'indicates that organizational pressure overrides job-context priming.}')
lines.append('\\label{tab:dimension-stats}')
lines.append('\\end{table}')

tex = '\n'.join(lines) + '\n'
out = TABLES_DIR / 'dimension_stats.tex'
out.write_text(tex)
print(f'Wrote {out}')
print()
print(tex)